# System prompt with Ollama models

This notebook demonstrates how to set up an [Ollama](https://ollama.com/) model and send a system prompt with Python in a Jupyter notebook.

## 1. Setup

To use this notebook, you need to have Ollama running locally. You can download it from [ollama.com](https://ollama.com/).

In [ ]:
!pip install requests

In [ ]:
import os
import time
import requests

OLLAMA_HOST = os.getenv("OLLAMA_HOST", "http://localhost:11434")
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "gemma3:4b")

def query_model(instructions: str | None,
                prompt: str,
                model: str = OLLAMA_MODEL,
                ) -> str:
    """Send a system instruction and a user prompt to a local LLM running through Ollama."""
    messages = []
    if instructions:
        messages.append({"role": "system", "content": instructions})
    messages.append({"role": "user", "content": prompt})

    payload = {
        "model": model,
        "messages": messages,
        "stream": False,
        "options": {
        },
    }

    start = time.perf_counter()
    response = requests.post(f"{OLLAMA_HOST}/api/chat",
                             json=payload,
                             timeout=120)
    latency = time.perf_counter() - start
    response.raise_for_status()
    result = response.json()

    input_tokens = result.get("prompt_eval_count", 0)
    output_tokens = result.get("eval_count", 0)
    print(f"\tModel: {result.get('model', model)}")
    print(f"\tLatency: {latency:.3f} seconds")
    print(f"\tInput tokens: {input_tokens}")
    print(f"\tOutput tokens: {output_tokens}")
    print(f"\tTotal tokens: {input_tokens + output_tokens}")

    return result["message"]["content"].strip()

In [ ]:
instructions = """
You are a strict grammar teacher.
Always respond in one sentence and correct any mistakes.
"""
prompt = "Explain me what is context engineering in simple words"
model = OLLAMA_MODEL

print("=== With system prompt ===")
response = query_model(instructions, prompt, model)
print("User query:", prompt)
print("Response:", response)

print("\n=== With only user prompt ===")
response = query_model(None, prompt, model)
print("User query:", prompt)
print("Response:", response)